# RAGTIME Single-Query Search

This notebook sends one query to the private RAGTIME `POST /pipeline` endpoint and displays the highest-scoring document IDs.

Decrypt the repository credentials before starting Jupyter:

```bash
sops decrypt --output .env config/secrets.sops.env
uv run jupyter lab notebooks/ragtime_single_query.ipynb
```

Change `QUERY` below, then run the cells in order.

In [ ]:
import os

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

BASE_URL = os.getenv("RAGTIME_BASE_URL", "").rstrip("/")
BEARER_TOKEN = os.getenv("RAGTIME_BEARER_TOKEN", "")
PIPELINE = "ragtime2%20"
QUERY = "health benefits and risks of Nordic walking for older adults"
TOP_K = 20
REQUEST_TIMEOUT_SECONDS = 30

if not BASE_URL or not BEARER_TOKEN:
    raise RuntimeError(
        "Missing RAGTIME credentials. Run: "
        "sops decrypt --output .env config/secrets.sops.env"
    )

In [ ]:
def authorization_value(token):
    token = token.strip()
    return token if token.lower().startswith("bearer ") else f"Bearer {token}"


def search_once(query, pipeline=PIPELINE):
    response = requests.post(
        f"{BASE_URL}/pipeline",
        headers={
            "Authorization": authorization_value(BEARER_TOKEN),
            "Content-Type": "application/json",
        },
        json={"pipeline": pipeline, "query": query},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()

    scores = payload.get("scores")
    if not isinstance(scores, dict):
        raise ValueError(f"Unexpected response keys: {sorted(payload)}")

    ranked = sorted(scores.items(), key=lambda item: (-float(item[1]), item[0]))
    results = pd.DataFrame(ranked, columns=["document_id", "score"])
    results.insert(0, "rank", range(1, len(results) + 1))
    return payload, results

In [ ]:
payload, results = search_once(QUERY)

print(f"Query: {QUERY}")
print(f"Pipeline: {payload.get('pipeline', PIPELINE)}")
print(f"Processed: {payload.get('processed')}")
print(f"Results returned: {len(results)}")
display(results.head(TOP_K))